# 2023 — F1 full-weekend data

Select **Python (primusEnv)**, then **Run All**. Downloads are enabled below. No package upgrades, background service or uploads are performed.

Available sessions: races, sprints, qualifying and practice; timing, entries, results, tyres, sectors, car telemetry, positions, recorded weather and control messages. Missing provider channels remain explicit. Race and sprint sessions are prioritised.

Output: `.local/f1_2025_2026/` (the existing shared archive, now accepting 2020–2026). Per-year progress is separate. The historical directory name is retained to keep the pilot and current UI working. Other F1 notebooks wait visibly for the same archive lock. Run `00_setup.ipynb` once first if sign-in is needed. Never enter credentials in a code cell.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'data/archive/store.py').is_file() and (p / 'primusEnv').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the F1 Primus AI project.')
expected = (ROOT / 'primusEnv/Scripts/python.exe').resolve()
if Path(sys.executable).resolve() != expected:
    raise RuntimeError(f'Select the primusEnv kernel: {expected}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Python:', sys.executable)


## Collection settings
The defaults collect the full eligible year without editing. 2026 is capped at October 26, 2026 and actual current time. Future/unfinished sessions stay out of the run.


In [ ]:
YEAR = 2023
THROUGH_DATE = '2026-10-26'  # Inclusive America/Chicago date; also bounded by current UTC time.
ROUND_NUMBERS = None        # None = every round; optional set such as {1, 2}.
RUN_DOWNLOAD = True         # Run All starts this collection; the agent has not run it.
DRY_RUN = False             # True = offline configuration preview only.

SESSION_LIMIT = None        # None = all eligible sessions; optional positive test limit.
RETRY_PARTIAL = False       # True explicitly revisits previously partial coverage.
AUTHENTICATE_LOCALLY = True  # Reuse SDK sign-in from 00_setup; True prompts locally again.


## Run / resume
Leave this cell running; it prints progress and provider cooldowns. Interrupt the kernel to stop safely. Run All again to resume and verify saved work. Access denial stops the run; sign in locally again if needed, then resume. Partial/unavailable data remain labelled. The data and HTTP accounting are shared across notebooks.


In [ ]:
if RUN_DOWNLOAD and not DRY_RUN:
    from data.archive.notebook_f1 import run_f1_year

    progress = run_f1_year(ROOT, YEAR, through=THROUGH_DATE, round_numbers=ROUND_NUMBERS,
                           session_limit=SESSION_LIMIT, retry_partial=RETRY_PARTIAL,
                           authenticate=AUTHENTICATE_LOCALLY)
    records = progress.get('sessions', {})
    if records:
        display(pd.DataFrame.from_dict(records, orient='index'))

else:
    print('Offline preview: no sign-in, requests, archive creation or download worker.')


## Next
Open `99_status_validate.ipynb` for an archive overview and integrity checks. Keep local caches and credentials out of Git/Kaggle. Publication is a later, separate step. See [the run guide](../../docs/data-collection-2020-2026.md).
